# Wine Quality Statistical Analysis

**Author: Feras Elghandour**

Quantify differences in measured properties across expert-rated quality groups while distinguishing association, effect size, and statistical uncertainty.

Descriptive analysis, wine-type stratification, correlation plots, directional Welch testing, an explicitly named standardized effect size, and a Welch confidence interval.

Read README.md for result scope and DATA.md for input preparation. Run cells in order in a clean environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scipy

print(f"numpy:      {np.__version__}")
print(f"pandas:     {pd.__version__}")
print(f"matplotlib: {plt.matplotlib.__version__}")
print(f"seaborn:    {sns.__version__}")
print(f"scipy:      {scipy.__version__}")

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 6)})

In [ ]:
df = pd.read_csv("wine_quality.csv")
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df.info()
print(f"\nWine type distribution:\n{df['wine_type'].value_counts()}")

In [ ]:
# Full summary statistics
desc = df.describe().round(3)
desc

In [ ]:
# Quality distribution
print("Quality rating distribution:")
print(df["quality"].value_counts().sort_index())
print(f"\nMean quality: {df['quality'].mean():.2f}")
print(f"Median quality: {df['quality'].median():.0f}")

In [ ]:
# Create quality tier for analysis
df["quality_tier"] = pd.cut(df["quality"], bins=[0, 5, 6, 10],
                            labels=["Low (3-5)", "Medium (6)", "High (7-9)"])
print("Quality tier distribution:")
print(df["quality_tier"].value_counts().sort_index())

In [ ]:
# Descriptive stats by wine type
wine_type_stats = df.groupby("wine_type")[["alcohol", "volatile acidity", "residual sugar",
                                            "pH", "sulphates", "quality"]].agg(["mean", "median", "std"]).round(3)
wine_type_stats

In [ ]:
# Descriptive stats by quality tier
quality_stats = df.groupby("quality_tier")[["alcohol", "volatile acidity",
    "residual sugar", "density", "pH"]].mean().round(3)
print("Mean chemical properties by quality tier:")
quality_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quality distribution
sns.countplot(data=df, x="quality", hue="wine_type",
              palette={"red": "#A84B2F", "white": "#20808D"}, ax=axes[0])
axes[0].set_title("Distribution of Quality Ratings by Wine Type", fontweight="bold")
axes[0].set_xlabel("Quality Rating (3-9)")
axes[0].set_ylabel("Count")
axes[0].legend(title="Wine Type")

# Alcohol distribution
sns.histplot(data=df, x="alcohol", hue="wine_type", kde=True,
             palette={"red": "#A84B2F", "white": "#20808D"}, alpha=0.5, ax=axes[1])
axes[1].set_title("Distribution of Alcohol Content by Wine Type", fontweight="bold")
axes[1].set_xlabel("Alcohol (% by volume)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title("Correlation Matrix — Chemical Properties and Quality", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
sns.boxplot(data=df, x="quality_tier", y="alcohol",
            palette=["#944454", "#FFC553", "#20808D"], ax=axes[0])
axes[0].set_title("Alcohol Content by Quality Tier", fontweight="bold")
axes[0].set_xlabel("Quality Tier")
axes[0].set_ylabel("Alcohol (% by volume)")

# Violin plot split by wine type
sns.violinplot(data=df, x="quality_tier", y="alcohol", hue="wine_type",
               split=True, palette={"red": "#A84B2F", "white": "#20808D"},
               inner="quartile", ax=axes[1])
axes[1].set_title("Alcohol by Quality Tier and Wine Type", fontweight="bold")
axes[1].set_xlabel("Quality Tier")
axes[1].set_ylabel("Alcohol (% by volume)")
axes[1].legend(title="Wine Type")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(df["alcohol"], df["volatile acidity"], c=df["quality"],
                     cmap="RdYlGn", alpha=0.4, s=15, edgecolors="none")
plt.colorbar(scatter, label="Quality Rating")
ax.set_title("Volatile Acidity vs. Alcohol Content (Colored by Quality)", fontweight="bold")
ax.set_xlabel("Alcohol (% by volume)")
ax.set_ylabel("Volatile Acidity (g/L)")
plt.tight_layout()
plt.show()

In [ ]:
# Define groups
high_quality = df[df["quality"] >= 7]["alcohol"]
low_quality = df[df["quality"] <= 5]["alcohol"]

print(f"High quality (7-9): n = {len(high_quality)}, mean = {high_quality.mean():.3f}, std = {high_quality.std():.3f}")
print(f"Low quality  (3-5): n = {len(low_quality)}, mean = {low_quality.mean():.3f}, std = {low_quality.std():.3f}")
print(f"Mean difference: {high_quality.mean() - low_quality.mean():.3f}% alcohol")

In [ ]:
# Welch's t-test (one-tailed: high > low)
t_stat, p_two_tailed = stats.ttest_ind(high_quality, low_quality, equal_var=False)
p_one_tailed = stats.ttest_ind(high_quality, low_quality, equal_var=False, alternative='greater').pvalue

print(f"Welch's t-statistic: {t_stat:.4f}")
print(f"p-value (two-tailed): {p_two_tailed:.2e}")
print(f"p-value (one-tailed): {p_one_tailed:.2e}")
print(f"Significance level:   α = 0.05")
print()
if t_stat > 0 and p_one_tailed < 0.05:
    print(f"Result: REJECT H₀ — High-quality wines have significantly higher alcohol (p = {p_one_tailed:.2e}).")
else:
    print(f"Result: FAIL TO REJECT H₀ (p = {p_one_tailed:.2e}).")

In [ ]:
# Effect size: Cohen's d
pooled_std = np.sqrt(((len(high_quality)-1)*high_quality.var() + (len(low_quality)-1)*low_quality.var()) / (len(high_quality)+len(low_quality)-2))
cohens_d = (high_quality.mean() - low_quality.mean()) / pooled_std
print(f"Cohen's d: {cohens_d:.3f}")
if abs(cohens_d) >= 0.8:
    print("Effect size: LARGE")
elif abs(cohens_d) >= 0.5:
    print("Effect size: MEDIUM")
elif abs(cohens_d) >= 0.2:
    print("Effect size: SMALL")
else:
    print("Effect size: NEGLIGIBLE")

In [ ]:
# 95% confidence interval for the difference in means
from scipy.stats import sem
diff = high_quality.mean() - low_quality.mean()
se_diff = np.sqrt(sem(high_quality)**2 + sem(low_quality)**2)
v_high = high_quality.var() / len(high_quality)
v_low = low_quality.var() / len(low_quality)
welch_df = (v_high + v_low)**2 / (v_high**2/(len(high_quality)-1) + v_low**2/(len(low_quality)-1))
critical = stats.t.ppf(0.975, welch_df)
ci_low = diff - critical * se_diff
ci_high = diff + critical * se_diff
print(f"95% CI for mean difference: [{ci_low:.3f}, {ci_high:.3f}]% alcohol")
print(f"Point estimate: {diff:.3f}% alcohol")